In [1]:
# Dataframe and Plotting libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Machine Learning libraries
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss


pd.set_option('display.max_columns', None)
# from google.colab import files
import warnings
warnings.filterwarnings('ignore')

In [122]:
# Load the files into a Pandas Dataframe
train = pd.read_csv('Train.csv')
test = pd.read_csv('Test.csv')
ss = pd.read_csv('SampleSubmission.csv')

In [123]:
print("Train shape: "+ str(train.shape))
print("Test shape: "+ str(test.shape))

Train shape: (373, 12)
Test shape: (558, 11)


In [6]:
train.head()

,MERCHANT_CATEGORIZED_AT,MERCHANT_NAME,MERCHANT_CATEGORIZED_AS,PURCHASE_VALUE,PURCHASED_AT,IS_PURCHASE_PAID_VIA_MPESA_SEND_MONEY,USER_AGE,USER_GENDER,USER_HOUSEHOLD,USER_INCOME,USER_ID,Transaction_ID
0,2022-05-04 10:25:50.588042+00,UONSDA CHURCH MOGERE MWAYO TITHE,Rent / Mortgage,1700,2022-05-04 13:56:00+00,False,NaN,Male,3,10000,ID_ZX4DCF4K,ID_04mk78fa
1,2021-10-25 16:18:38.586837+00,PARK N GO,Transport & Fuel,100,2021-10-24 14:12:00+00,False,25.0,Female,4,90000,ID_U9WZMGJZ,ID_04xkfb07
2,2022-05-20 14:17:30.917297+00,SAFARICOM OFFERS TUNUKIWA,Data & WiFi,20,2022-05-27 12:46:00+00,False,NaN,Male,3,10000,ID_ZX4DCF4K,ID_051urwuw
3,2022-04-12 15:59:14.139347+00,ZILLIONS CREDIT LIMITED,Loan Repayment,3800,2022-02-07 12:16:00+00,False,NaN,Male,5,252000,ID_3JA0MAFB,ID_0e1om7rz
4,2022-05-04 10:24:26.709045+00,MICHAEL MUSEMBI,Bills & Fees,60,2022-05-04 15:28:00+00,False,NaN,Male,3,10000,ID_ZX4DCF4K,ID_0kfcoawb


In [7]:
test.head()

,MERCHANT_CATEGORIZED_AT,MERCHANT_NAME,PURCHASE_VALUE,PURCHASED_AT,IS_PURCHASE_PAID_VIA_MPESA_SEND_MONEY,USER_AGE,USER_GENDER,USER_HOUSEHOLD,USER_INCOME,USER_ID,Transaction_ID
0,2022-06-01 10:25:16.7131+00,KCB PAYBILL AC,150000,2022-05-05 08:29:00+00,True,NaN,Male,5,150000,ID_O8P8YS18,ID_00x9h2yx
1,2022-03-16 13:05:51.851102+00,IPAY LTD,7394,2019-10-05 16:02:00+00,False,NaN,Female,1,10000,ID_40L9OTIM,ID_01db594f
2,2022-03-10 18:25:42.112294+00,JOHN NJIRU GAKUNJU,200,2021-11-25 09:56:00+00,False,NaN,Female,1,85000,ID_1I8XYBWK,ID_07pm99no
3,2022-04-12 15:59:14.139347+00,UONSDA CHURCH OFFERING-MAIN,50,2022-04-11 14:48:00+00,False,NaN,Male,2,10000,ID_GR569FUO,ID_09jryhkd
4,2022-05-04 10:24:26.709045+00,M-SHWARI LOCK SAVINGS,10000,2022-05-04 13:59:00+00,False,NaN,Male,3,10000,ID_ZX4DCF4K,ID_0ccv924c


# Prepare

In [127]:
test.isna().sum()

MERCHANT_CATEGORIZED_AT                    0
MERCHANT_NAME                              0
PURCHASE_VALUE                             0
PURCHASED_AT                               0
IS_PURCHASE_PAID_VIA_MPESA_SEND_MONEY      0
USER_AGE                                 473
USER_GENDER                                5
USER_HOUSEHOLD                             0
USER_INCOME                                0
USER_ID                                    0
Transaction_ID                             0
dtype: int64

In [131]:
features = train[['MERCHANT_CATEGORIZED_AT',"PURCHASE_VALUE","PURCHASED_AT","IS_PURCHASE_PAID_VIA_MPESA_SEND_MONEY","USER_GENDER","USER_INCOME","USER_HOUSEHOLD"]]

In [132]:
target = train["MERCHANT_CATEGORIZED_AS"]

In [133]:
features["MERCHANT_CATEGORIZED_AT"] = pd.to_datetime(features["MERCHANT_CATEGORIZED_AT"])
features["MERCHANT_CATEGORIZED_AT"] = features['MERCHANT_CATEGORIZED_AT'].dt.hour

In [134]:
features["PURCHASED_AT"] = pd.to_datetime(features["PURCHASED_AT"])
features["PURCHASED_AT"] = features['PURCHASED_AT'].dt.hour

In [135]:
def binate(x):
    if x == True:
        return 1
    elif x == False:
        return 0
    elif x == "Male":
        return 1
    elif x == "Female":
        return 0
    else:
        return 1

In [136]:
features["IS_PURCHASE_PAID_VIA_MPESA_SEND_MONEY"] = features["IS_PURCHASE_PAID_VIA_MPESA_SEND_MONEY"].apply(binate)

In [137]:
features["USER_GENDER"] = features["USER_GENDER"].apply(binate)

In [138]:
features.head()

,MERCHANT_CATEGORIZED_AT,PURCHASE_VALUE,PURCHASED_AT,IS_PURCHASE_PAID_VIA_MPESA_SEND_MONEY,USER_GENDER,USER_INCOME,USER_HOUSEHOLD
0,10,1700,13,0,1,10000,3
1,16,100,14,0,0,90000,4
2,14,20,12,0,1,10000,3
3,15,3800,12,0,1,252000,5
4,10,60,15,0,1,10000,3


In [139]:
features.isna().sum()

MERCHANT_CATEGORIZED_AT                  0
PURCHASE_VALUE                           0
PURCHASED_AT                             0
IS_PURCHASE_PAID_VIA_MPESA_SEND_MONEY    0
USER_GENDER                              0
USER_INCOME                              0
USER_HOUSEHOLD                           0
dtype: int64

# Prepare TEST

In [140]:
tfeatures = test[['MERCHANT_CATEGORIZED_AT',"PURCHASE_VALUE","PURCHASED_AT","IS_PURCHASE_PAID_VIA_MPESA_SEND_MONEY","USER_GENDER","USER_INCOME","USER_HOUSEHOLD"]]

In [141]:
tfeatures["MERCHANT_CATEGORIZED_AT"] = pd.to_datetime(tfeatures["MERCHANT_CATEGORIZED_AT"])
tfeatures["MERCHANT_CATEGORIZED_AT"] = tfeatures['MERCHANT_CATEGORIZED_AT'].dt.hour

In [142]:
tfeatures["PURCHASED_AT"] = pd.to_datetime(tfeatures["PURCHASED_AT"])
tfeatures["PURCHASED_AT"] = tfeatures['PURCHASED_AT'].dt.hour

In [143]:
tfeatures["IS_PURCHASE_PAID_VIA_MPESA_SEND_MONEY"] = tfeatures["IS_PURCHASE_PAID_VIA_MPESA_SEND_MONEY"].apply(binate)

In [144]:
tfeatures["USER_GENDER"] = tfeatures["USER_GENDER"].apply(binate)

In [145]:
tfeatures.head()

,MERCHANT_CATEGORIZED_AT,PURCHASE_VALUE,PURCHASED_AT,IS_PURCHASE_PAID_VIA_MPESA_SEND_MONEY,USER_GENDER,USER_INCOME,USER_HOUSEHOLD
0,10,150000,8,1,1,150000,5
1,13,7394,16,0,0,10000,1
2,18,200,9,0,0,85000,1
3,15,50,14,0,1,10000,2
4,10,10000,13,0,1,10000,3


In [146]:
tfeatures.isna().sum()

MERCHANT_CATEGORIZED_AT                  0
PURCHASE_VALUE                           0
PURCHASED_AT                             0
IS_PURCHASE_PAID_VIA_MPESA_SEND_MONEY    0
USER_GENDER                              0
USER_INCOME                              0
USER_HOUSEHOLD                           0
dtype: int64

In [147]:
tfeatures.shape

(558, 7)

# MODEL

In [148]:
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

In [149]:
svm = LinearSVC()
clf = CalibratedClassifierCV(svm) 
clf.fit(features, target)
predictions = clf.predict_proba(tfeatures)

In [150]:
predictions.shape

(558, 13)

In [152]:
predictions[0:3]

array([[0.1296543 , 0.11654798, 0.04837346, 0.03214781, 0.18225702,
        0.09645058, 0.09961173, 0.01635732, 0.02354946, 0.0927924 ,
        0.01250895, 0.04824757, 0.10150143],
       [0.22447794, 0.10800765, 0.01354965, 0.04306947, 0.09560816,
        0.10309916, 0.12465411, 0.02324587, 0.01749667, 0.11208675,
        0.01741196, 0.05741986, 0.05987276],
       [0.20067226, 0.11279423, 0.01399771, 0.03802428, 0.09761244,
        0.10707713, 0.13070635, 0.02342032, 0.01935832, 0.10875942,
        0.01622583, 0.05582404, 0.07552765]])

In [154]:
clf.classes_

array(['Bills & Fees', 'Data & WiFi', 'Education', 'Emergency fund',
       'Family & Friends', 'Going out', 'Groceries', 'Health',
       'Loan Repayment', 'Miscellaneous', 'Rent / Mortgage', 'Shopping',
       'Transport & Fuel'], dtype=object)

In [157]:
predictions_df = pd.DataFrame(predictions)
predictions_df.columns = clf.classes_
predictions_df.head()


,Bills & Fees,Data & WiFi,Education,Emergency fund,Family & Friends,Going out,Groceries,Health,Loan Repayment,Miscellaneous,Rent / Mortgage,Shopping,Transport & Fuel
0,0.129654,0.116548,0.048373,0.032148,0.182257,0.096451,0.099612,0.016357,0.023549,0.092792,0.012509,0.048248,0.101501
1,0.224478,0.108008,0.013550,0.043069,0.095608,0.103099,0.124654,0.023246,0.017497,0.112087,0.017412,0.057420,0.059873
2,0.200672,0.112794,0.013998,0.038024,0.097612,0.107077,0.130706,0.023420,0.019358,0.108759,0.016226,0.055824,0.075528
3,0.226331,0.106940,0.012974,0.042626,0.087204,0.101544,0.129014,0.023541,0.017126,0.110133,0.017262,0.056628,0.068679
4,0.223391,0.108160,0.013754,0.043127,0.098559,0.103427,0.123219,0.023107,0.017591,0.112572,0.017428,0.057626,0.058039


In [156]:
ss.head()

,Transaction_ID,Bills & Fees,Data & WiFi,Education,Emergency fund,Family & Friends,Going out,Groceries,Health,Loan Repayment,Miscellaneous,Rent / Mortgage,Shopping,Transport & Fuel
0,ID_00x9h2yx,0,0,0,0,0,0,0,0,0,0,0,0,0
1,ID_01db594f,0,0,0,0,0,0,0,0,0,0,0,0,0
2,ID_07pm99no,0,0,0,0,0,0,0,0,0,0,0,0,0
3,ID_09jryhkd,0,0,0,0,0,0,0,0,0,0,0,0,0
4,ID_0ccv924c,0,0,0,0,0,0,0,0,0,0,0,0,0


In [158]:
predictions_df['Transaction_ID'] = ss['Transaction_ID']
predictions_df.head()

,Bills & Fees,Data & WiFi,Education,Emergency fund,Family & Friends,Going out,Groceries,Health,Loan Repayment,Miscellaneous,Rent / Mortgage,Shopping,Transport & Fuel,Transaction_ID
0,0.129654,0.116548,0.048373,0.032148,0.182257,0.096451,0.099612,0.016357,0.023549,0.092792,0.012509,0.048248,0.101501,ID_00x9h2yx
1,0.224478,0.108008,0.013550,0.043069,0.095608,0.103099,0.124654,0.023246,0.017497,0.112087,0.017412,0.057420,0.059873,ID_01db594f
2,0.200672,0.112794,0.013998,0.038024,0.097612,0.107077,0.130706,0.023420,0.019358,0.108759,0.016226,0.055824,0.075528,ID_07pm99no
3,0.226331,0.106940,0.012974,0.042626,0.087204,0.101544,0.129014,0.023541,0.017126,0.110133,0.017262,0.056628,0.068679,ID_09jryhkd
4,0.223391,0.108160,0.013754,0.043127,0.098559,0.103427,0.123219,0.023107,0.017591,0.112572,0.017428,0.057626,0.058039,ID_0ccv924c


In [ ]:
# Create the submission csv file
sub.to_csv('first_submission.csv', index = False)